In [2]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


In [3]:
import requests

headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}

payload = {
    "model": "gpt-6-luna",
    "reasoning_effort": "none",
    "messages": [
        {"role": "user", "content": "Tell me a fun fact"}]
}

payload

{'model': 'gpt-6-luna',
 'reasoning_effort': 'none',
 'messages': [{'role': 'user', 'content': 'Tell me a fun fact'}]}

In [4]:
response = requests.post(
    "https://api.openai.com/v1/chat/completions",
    headers=headers,
    json=payload
)

response.json()

{'id': 'chatcmpl-EfCtwd7lZUj9LMz5DIqUZtACHhr7p',
 'object': 'chat.completion',
 'created': 1793489748,
 'model': 'gpt-6-luna',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': 'Octopuses have three hearts: two pump blood to the gills, and one pumps it to the rest of the body. The heart serving the body actually stops beating while the octopus swims—which may be one reason they prefer crawling.',
    'refusal': None,
    'annotations': []},
   'finish_reason': 'stop'}],
 'usage': {'prompt_tokens': 11,
  'completion_tokens': 51,
  'total_tokens': 62,
  'prompt_tokens_details': {'cached_tokens': 0,
   'cache_write_tokens': 0,
   'audio_tokens': 0},
  'completion_tokens_details': {'reasoning_tokens': 0,
   'audio_tokens': 0,
   'accepted_prediction_tokens': 0,
   'rejected_prediction_tokens': 0}},
 'service_tier': 'default',
 'system_fingerprint': None}

In [5]:
response.json()["choices"][0]["message"]["content"]

'Octopuses have three hearts: two pump blood to the gills, and one pumps it to the rest of the body. The heart serving the body actually stops beating while the octopus swims—which may be one reason they prefer crawling.'

In [6]:
from openai import OpenAI
openai = OpenAI()

response = openai.chat.completions.create(model="gpt-6-luna", reasoning_effort="none", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

'Octopuses have three hearts: two pump blood to the gills, while the third pumps it to the rest of the body. Their blood is blue, too, because it uses copper-rich hemocyanin to carry oxygen.'

In [7]:
requests.get("http://localhost:11434").content

b'Ollama is running'

In [8]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [9]:
# Get a fun fact

response = ollama.chat.completions.create(model="llama3.2", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

'Here\'s a fun fact:\n\nDid you know that there is a species of jellyfish that is immortal?! The Turritopsis dohrnii, also known as the "immortal jellyfish," can transform its body into a younger state through a process called transdifferentiation, effectively making it immune to death. It can revert back into its polyp stage, which is the juvenile form of a jellyfish, and then grow back into an adult again. Isn\'t that just mind-blowing?'

In [10]:
response = ollama.chat.completions.create(model="deepseek-r1:1.5b", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

'\n\nSure! Here\'s a fun fact for you: **"Eiffel Tower Day is Friday, November 25th. The famous Eiffel Tower will be climbed for the second time during the day to celebrate its impressive 330-year heritage"! \n\nThat\'s pretty cool! 😊'

In [14]:
system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""
user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""


In [19]:
import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")


website = fetch_website_contents("https://www.theverge.com/")
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
def summarize(url):
    website = fetch_website_contents(url)
    response = ollama.chat.completions.create(
        model = "deepseek-r1:1.5b",
        reasoning_effort = "none",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [20]:
summarize("https://www.theverge.com/")

'\n\nThe Verge  \n\nCheck *Your* Statuses  \n\nThe Verge logo  \nSkip to main content  \n\nThe Verge  \nThe Verge logo.  \nTech  \nReviews  \nScience  \nEntertainment  \nAI  \nPolicy  \nNotifications  \n\nNotifications  \nNotifications  \nHamburger Navigation Button  \nNotifications  \nNotifications  \nHamburger Navigation Button  \nNavigation Drawer  \nThe Verge  \nThe Verge logo.  \nLogin  \n/  \nSign Up  \nclose  \nClose  \nSearch  \nLight  \nSystem  \nDark  \nTech  \nExpand  \nAmazon  \nApple  \nFacebook  \nGoogle  \nMicrosoft  \nSamsung  \nBusiness  \nSee all tech  \nReviews  \nExpand  \nSmart Home Reviews  \nPhone Reviews  \nTablet Reviews  \nHeadphone Reviews  \nSee all reviews  \nScience  \nExpand  \nSpace  \nEnergy  \nEnvironment  \nHealth  \nSee all science  \nEntertainment  \nExpand  \nTV Shows  \nMovies  \nAudio  \nSee all entertainment  \nAI  \nExpand  \nOpenAI  \nAnthropic  \nSee all AI  \nPolicy  \nExpand  \nAntitrust  \nPolitics  \nLaw  \nSecurity  \nSee all policy  \nG

In [21]:
def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [22]:
display_summary("https://www.theverge.com/")



The website contains a wealth of tech-related content, covering reviews, news, and updates on a wide range of tech topics, including Space, Energy, and Health. Here are some highlights:

- **Tech News**: Breakers and updates on the latest tech advancements and challenges.
- ** Reviews**: Tech-related reviews on products and services across different platforms.
- **Blogs**: Tech-generated content, humorously curated and written by the Verge team.

Let me know if you'd like recommendations for fun tech-related reads or something!